# Analyse de convergence des PINNs — plan factoriel 2⁴ (16 modèles)

Étude d'ablation du PINN de ballottement (`SINN`) selon 4 features binaires :

| Code  | Feature                        | Flag `Config`                  | Artefacts spécifiques |
|-------|--------------------------------|--------------------------------|-----------------------|
| `ff`  | Fourier features               | `use_fourier_features`         | — |
| `as`  | Échantillonnage adaptatif      | `use_adaptive_sampling`        | `adaptive_history.npz/json` |
| `ts`  | Segmentation temporelle        | `use_temporal_segmentation`    | colonnes `window`, `t_left`, `t_right` |
| `alw` | Poids adaptatifs (GradNorm)    | `use_adaptive_loss_weights`    | `adaptive_weights.json` |

Convention de nommage : `model_ff{0,1}_as{0,1}_ts{0,1}_alw{0,1}` dans `runs/`.

**Trois précautions méthodologiques qui structurent le notebook**

1. **La loss `total` n'est pas comparable d'un run à l'autre** : elle dépend des poids (GradNorm les modifie),
   des points de collocation (l'échantillonnage adaptatif les déplace vers les zones difficiles, donc la loss
   *augmente* mécaniquement) et de la fenêtre courante (segmentation). On utilise donc :
   - une loss `core` non pondérée (continuité + VOF + quantité de mouvement + parois), présente dans toutes les fenêtres ;
   - surtout une **métrique hold-out** : résidus physiques du modèle final sur une grille (x, y, t) **identique pour les 16 modèles**.
2. **Une seule graine par configuration** → aucune barre d'erreur : les effets estimés sont des ordres de grandeur,
   pas des résultats statistiquement établis. La section 0 permet de lancer plusieurs graines.
3. **Un résidu faible ne prouve pas une solution correcte** (solution triviale possible) : la section 8 compare à OpenFOAM si le jeu de données est fourni.

## 0. Configuration et (optionnel) entraînement des 16 modèles

In [ ]:
import itertools
import json
from dataclasses import fields as dc_fields, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from visualization import (
    build_sinn_data_from_model,
    load_model,
    plot_histories,
    plot_history,
    plot_residuals,
)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.float_format", "{:.3e}".format)
pd.set_option("display.width", 200)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUNS_DIR = Path("runs")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
EPOCHS = 5000

FACTORS = ["ff", "as", "ts", "alw"]
FEATURE_NAMES = {
    "ff": "Fourier features",
    "as": "Adaptive sampling",
    "ts": "Temporal segmentation",
    "alw": "Adaptive weights (GradNorm)",
}
CFG_FLAGS = {
    "ff": "use_fourier_features",
    "as": "use_adaptive_sampling",
    "ts": "use_temporal_segmentation",
    "alw": "use_adaptive_loss_weights",
}

# Termes de loss
LOSS_TERMS = ["continuity", "vof", "momentum_x", "momentum_y",
              "initial_u", "initial_v", "initial_alpha",
              "wall_u", "wall_v", "pressure_gauge"]
PHYS = ["continuity", "vof", "momentum_x", "momentum_y"]
CORE = PHYS + ["wall_u", "wall_v"]   # présents dans toutes les fenêtres


def run_name(combo):
    return "model_ff{}_as{}_ts{}_alw{}".format(*combo)


def short(name):
    return name.replace("model_", "")


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


COMBOS = list(itertools.product([0, 1], repeat=4))
RUNS = pd.DataFrame(
    [{**dict(zip(FACTORS, c)), "name": run_name(c), "dir": RUNS_DIR / run_name(c)}
     for c in COMBOS]
).set_index("name")
RUNS

In [ ]:
# --- Entraînement (désactivé par défaut) -----------------------------
RUN_TRAINING = False
OVERWRITE = False
SEEDS = [None]          # ex. [0, 1, 2] -> runs model_..._seed{s} (nécessite d'adapter RUNS)

if RUN_TRAINING:
    from config import Config
    from train import train

    base_cfg = Config()
    for name, row in RUNS.iterrows():
        if (row["dir"] / "history.json").exists() and not OVERWRITE:
            print(f"skip {name} (déjà entraîné)")
            continue
        cfg = replace(base_cfg, **{CFG_FLAGS[k]: bool(row[k]) for k in FACTORS})
        train(cfg=cfg, output_dir=RUNS_DIR, model_name=name, epochs=EPOCHS)

## 1. Chargement des historiques et contrôle de cohérence

In [ ]:
HIST, CFGS = {}, {}
for name, row in RUNS.iterrows():
    hp, cp = row["dir"] / "history.json", row["dir"] / "config.json"
    if hp.exists():
        HIST[name] = pd.DataFrame(json.loads(hp.read_text(encoding="utf-8")))
    if cp.exists():
        CFGS[name] = json.loads(cp.read_text(encoding="utf-8"))

missing = [n for n in RUNS.index if n not in HIST]
print(f"{len(HIST)}/16 runs chargés" + (f" — manquants : {missing}" if missing else ""))

for name, df in HIST.items():
    df["core"] = df[CORE].sum(axis=1)
    df["phys"] = df[PHYS].sum(axis=1)

# Contrôle : les flags de config.json correspondent bien au nom du run
rows = []
for name, df in HIST.items():
    row = {"run": short(name), "epochs": int(df["epoch"].max()),
           "n_windows": int(df["window"].nunique())}
    if name in CFGS:
        for k in FACTORS:
            flag = bool(CFGS[name].get(CFG_FLAGS[k]))
            row[f"{k}_ok"] = flag == bool(RUNS.loc[name, k])
    rows.append(row)
CHECK = pd.DataFrame(rows).set_index("run")
CHECK

## 2. Vue d'ensemble de la convergence

### 2.1 Tous les termes de loss, run par run
(utilise `plot_history` ; les lignes plates en bas = termes initiaux nuls hors fenêtre 0)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 14), sharex=True, sharey=True)
for ax, (name, row) in zip(axes.ravel(), RUNS.iterrows()):
    if name not in HIST:
        ax.set_title(short(name) + " (absent)", fontsize=9)
        continue
    plot_history(row["dir"] / "history.json", ax=ax, legend=False,
                 title=short(name), loss_names=LOSS_TERMS)
    ax.title.set_fontsize(9)
    # frontières de fenêtres (segmentation temporelle)
    if row["ts"] == 1:
        starts = HIST[name].groupby("window")["epoch"].min().iloc[1:]
        for e in starts:
            ax.axvline(e, color="k", lw=0.4, alpha=0.4)
for ax in axes.ravel():
    ax.set_ylim(1e-9, None)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5)
fig.tight_layout(rect=[0, 0.05, 1, 1])
savefig(fig, "01_all_terms_per_run")

### 2.2 Loss totale brute (comparaison « naïve »)
À lire avec prudence — voir précaution n°1.

In [ ]:
paths = [RUNS.loc[n, "dir"] / "history.json" for n in HIST]
fig, ax = plot_histories(paths, names=[short(n) for n in HIST])
fig.set_size_inches(14, 6)
savefig(fig, "02_total_loss_raw")

### 2.3 Loss `core` non pondérée, colorée par présence de chaque feature
Rouge = feature activée, bleu = désactivée (médiane glissante de 50 epochs).

In [ ]:
def smooth(y, w=50):
    return pd.Series(np.asarray(y, dtype=float)).rolling(w, min_periods=1).median().to_numpy()


fig, axes = plt.subplots(1, 4, figsize=(22, 4.5), sharey=True)
for ax, k in zip(axes, FACTORS):
    for name, df in HIST.items():
        on = RUNS.loc[name, k] == 1
        ax.semilogy(df["epoch"], smooth(df["core"]), color="C3" if on else "C0",
                    alpha=0.55, lw=1.2)
    ax.plot([], [], color="C0", label=f"{k}=0")
    ax.plot([], [], color="C3", label=f"{k}=1")
    ax.set_title(FEATURE_NAMES[k])
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set_ylabel("Loss core non pondérée")
fig.tight_layout()
savefig(fig, "03_core_by_factor")

## 3. Métriques de convergence extraites de l'historique

* `core_final` : moyenne de la loss `core` sur les 100 dernières epochs ;
* `decades_gained` : décades gagnées entre les 100 premières et 100 dernières epochs ;
* `osc` : écart-type (en décades) des fluctuations autour de la tendance sur les 500 dernières epochs — mesure la stabilité ;
* `ep_1dec`, `ep_2dec` : epoch où la loss lissée a perdu 1 / 2 décades (vitesse de convergence).

> Pour `ts=1`, la loss `core` est évaluée sur la **fenêtre courante** : `core_final` ne concerne que la dernière fenêtre.
> Ces colonnes sont donc surtout utiles pour comparer à features égales ; la comparaison globale se fait en section 4.

In [ ]:
def epochs_to_drop(df, decades, n0=100):
    y = smooth(df["core"].to_numpy(), 25)
    thr = df["core"].iloc[:n0].mean() * 10.0 ** (-decades)
    idx = np.flatnonzero(y <= thr)
    return float(df["epoch"].iloc[idx[0]]) if idx.size else np.nan


def history_metrics(df, n_last=100):
    core = df["core"].to_numpy()
    lg = np.log10(np.maximum(core[-500:], 1e-16))
    trend = pd.Series(lg).rolling(25, min_periods=1, center=True).mean().to_numpy()
    out = {
        "core_final": core[-n_last:].mean(),
        "core_min": core.min(),
        "decades_gained": np.log10(core[:n_last].mean() / core[-n_last:].mean()),
        "osc": float(np.std(lg - trend)),
        "ep_1dec": epochs_to_drop(df, 1),
        "ep_2dec": epochs_to_drop(df, 2),
    }
    out.update({f"{k}_final": df[k].iloc[-n_last:].mean() for k in CORE})
    return out


METRICS = RUNS[FACTORS].join(
    pd.DataFrame.from_dict({n: history_metrics(d) for n, d in HIST.items()}, orient="index")
)
METRICS.index = METRICS.index.map(short)
METRICS[FACTORS + ["core_final", "decades_gained", "osc", "ep_1dec", "ep_2dec"]].sort_values("core_final")

## 4. Métrique hold-out : résidus physiques sur une grille commune

Chaque modèle final (`checkpoint_final.pt`) est évalué sur **la même grille (x, y, t)** couvrant tout l'horizon,
avec les **résidus non pondérés** (`model.compute_residuals`). C'est la seule comparaison réellement équitable entre les 16 configurations,
et elle révèle aussi la dérive temporelle (oubli) des modèles segmentés.

Cette section est ignorée pour les runs dont le checkpoint est absent.

In [ ]:
N_EVAL = dict(nx=50, ny=25, nt=30)
BATCH = 10_000


def load_cfg(name):
    from config import Config
    names = {f.name for f in dc_fields(Config)}
    cfg = Config(**{k: v for k, v in CFGS[name].items() if k in names})
    if hasattr(cfg, "device"):
        cfg = replace(cfg, device=DEVICE)
    return cfg


def load_run_model(name):
    d = RUNS.loc[name, "dir"]
    ckpt, cfg = d / "checkpoint_final.pt", load_cfg(name)
    try:
        model = load_model(checkpoint=str(ckpt), config=cfg, device=DEVICE)
    except Exception as exc:  # checkpoint complet (dict avec la clé "model")
        print(f"  load_model a échoué ({type(exc).__name__}) -> fallback state_dict")
        from model import SloshingPINN
        model = SloshingPINN(config=cfg).to(DEVICE)
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state["model"] if "model" in state else state)
    model.eval()
    return model


def eval_points(cfg, nx, ny, nt):
    x = np.linspace(float(cfg.x_min), float(cfg.x_max), nx)
    y = np.linspace(float(cfg.y_min), float(cfg.y_max), ny)
    t = np.linspace(float(cfg.t_min), float(cfg.t_max), nt)
    T, Y, X = np.meshgrid(t, y, x, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), T.ravel()], axis=1).astype(np.float32)
    return t, coords


def residual_by_time(model, coords, nt):
    sq = {k: [] for k in PHYS}
    for i in range(0, len(coords), BATCH):
        c = torch.as_tensor(coords[i:i + BATCH], dtype=torch.float32, device=DEVICE)
        with torch.enable_grad():
            res = model.compute_residuals(c)
        for k in PHYS:
            sq[k].append(res[k].detach().cpu().numpy().ravel() ** 2)
    return {k: np.concatenate(v).reshape(nt, -1).mean(axis=1) for k, v in sq.items()}


RES = {}
available = [n for n in RUNS.index if n in CFGS and (RUNS.loc[n, "dir"] / "checkpoint_final.pt").exists()]
if available:
    t_eval, coords_eval = eval_points(load_cfg(available[0]), **N_EVAL)
    for name in available:
        print("évaluation", short(name))
        model = load_run_model(name)
        RES[name] = {"t": t_eval, **residual_by_time(model, coords_eval, N_EVAL["nt"])}
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    HOLD = pd.DataFrame({
        short(n): {**{f"res_{k}": r[k].mean() for k in PHYS},
                   "res_phys": sum(r[k].mean() for k in PHYS)}
        for n, r in RES.items()
    }).T
    METRICS = METRICS.drop(columns=[c for c in HOLD.columns if c in METRICS.columns]).join(HOLD)
    display(METRICS[FACTORS + list(HOLD.columns)].sort_values("res_phys"))
else:
    print("Aucun checkpoint_final.pt trouvé : section hold-out ignorée.")

In [ ]:
if RES:
    fig, ax = plt.subplots(figsize=(13, 5))
    for name, r in RES.items():
        phys_t = sum(r[k] for k in PHYS)
        ax.semilogy(r["t"], phys_t, lw=1.3, label=short(name))
    ax.set_xlabel("t [s]")
    ax.set_ylabel("Résidu physique moyen² (grille commune)")
    ax.set_title("Résidu hold-out en fonction du temps")
    ax.legend(ncol=2, fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
    fig.tight_layout()
    savefig(fig, "04_holdout_residual_vs_time")

## 5. Effets factoriels : quelle feature compte, et quelles interactions ?

On ajuste sur `y = log10(métrique)` un modèle avec les 4 effets principaux et les 6 interactions d'ordre 2 (codage ±1).
Pour un plan complet 2⁴, l'effet d'une feature est la **différence moyenne de `log10(métrique)` entre « activée » et « désactivée »**
(valeur négative = la feature réduit l'erreur) ; `×facteur = 10^effet`.

⚠️ Sans répétition (1 graine), pas d'intervalle de confiance : la colonne `résidu rms` donne l'ordre de grandeur de ce qui
n'est pas expliqué par les effets d'ordre ≤ 2 (interactions d'ordre 3-4 + bruit d'entraînement). Un effet inférieur à ce niveau n'est pas fiable.

In [ ]:
def effects_table(metric, max_order=2):
    d = METRICS.dropna(subset=[metric])
    if len(d) < 8:
        return None, np.nan
    y = np.log10(d[metric].to_numpy(dtype=float))
    x = 2 * d[FACTORS].to_numpy(dtype=float) - 1
    terms, cols = [], [np.ones(len(d))]
    for r in range(1, max_order + 1):
        for combo in itertools.combinations(range(4), r):
            terms.append("×".join(FACTORS[i] for i in combo))
            cols.append(np.prod(x[:, combo], axis=1))
    A = np.column_stack(cols)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid_rms = float(np.sqrt(np.mean((y - A @ coef) ** 2)))
    eff = pd.Series(2 * coef[1:], index=terms, name=metric)
    return eff, resid_rms


metric_list = [m for m in ["core_final", "res_phys", "osc", "ep_1dec"] if m in METRICS.columns]
effects, rms = {}, {}
for m in metric_list:
    effects[m], rms[m] = effects_table(m)
EFF = pd.DataFrame({m: e for m, e in effects.items() if e is not None})
print("résidu rms du modèle d'effets (décades) :", {m: round(v, 3) for m, v in rms.items()})
display(EFF)
display((10 ** EFF).rename(columns=lambda c: f"{c} (×)"))

In [ ]:
TARGET = "res_phys" if "res_phys" in EFF.columns else "core_final"
e = EFF[TARGET].sort_values(key=np.abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(e.index, e.values, color=["C2" if v < 0 else "C3" for v in e.values])
ax.axvline(0, color="k", lw=0.8)
ax.axvspan(-rms[TARGET], rms[TARGET], color="gray", alpha=0.2, label="résidu rms (bruit + ordre ≥ 3)")
for i, v in enumerate(e.values):
    ax.text(v, i, f" ×{10**v:.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=8)
ax.set_xlabel(f"Effet sur log10({TARGET})   (vert = améliore)")
ax.set_title("Diagramme de Pareto des effets")
ax.legend(loc="lower right")
fig.tight_layout()
savefig(fig, "05_effects_pareto")

In [ ]:
# Heatmap 4x4 : lignes (ff, as) × colonnes (ts, alw)
d = METRICS.reset_index().rename(columns={"index": "run"})
d["row"] = d.apply(lambda r: f"ff={int(r['ff'])} as={int(r['as'])}", axis=1)
d["col"] = d.apply(lambda r: f"ts={int(r['ts'])} alw={int(r['alw'])}", axis=1)
fig, axes = plt.subplots(1, len(metric_list[:2]), figsize=(6.5 * len(metric_list[:2]), 4.5))
axes = np.atleast_1d(axes)
for ax, m in zip(axes, metric_list[:2]):
    P = d.pivot(index="row", columns="col", values=m)
    L = np.log10(P.astype(float))
    im = ax.imshow(L.values, cmap="viridis_r")
    ax.set_xticks(range(L.shape[1]), L.columns, rotation=30)
    ax.set_yticks(range(L.shape[0]), L.index)
    for i in range(L.shape[0]):
        for j in range(L.shape[1]):
            ax.text(j, i, f"{P.values[i, j]:.1e}", ha="center", va="center", color="w", fontsize=8)
    ax.set_title(f"{m} (couleur = log10)")
    ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
savefig(fig, "06_heatmap_metrics")

## 6. Fourier features (`ff`) — comparaison appariée

Pour chacun des 8 contextes (`as`, `ts`, `alw`), rapport de la métrique `ff=1 / ff=0` (< 1 : les Fourier features aident).

In [ ]:
def paired_ratio(metric, factor):
    others = [f for f in FACTORS if f != factor]
    rows = []
    for ctx, g in METRICS.groupby(others):
        g = g.dropna(subset=[metric])
        if set(g[factor]) == {0, 1}:
            a = g[g[factor] == 1][metric].iloc[0]
            b = g[g[factor] == 0][metric].iloc[0]
            rows.append({"contexte": " ".join(f"{k}={int(v)}" for k, v in zip(others, ctx)),
                         "ratio": a / b})
    return pd.DataFrame(rows).set_index("contexte")["ratio"] if rows else None


def plot_paired(factor, metrics):
    metrics = [m for m in metrics if m in METRICS.columns]
    if not metrics:
        return None
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, metrics):
        r = paired_ratio(m, factor)
        if r is None:
            continue
        ax.barh(r.index, np.log10(r.values), color=["C2" if v < 1 else "C3" for v in r.values])
        ax.axvline(0, color="k", lw=0.8)
        ax.set_xlabel("log10(ratio on/off)  (< 0 : la feature aide)")
        ax.set_title(f"{FEATURE_NAMES[factor]} — {m}")
    fig.tight_layout()
    return fig


fig = plot_paired("ff", ["core_final", "res_phys", "ep_1dec"])
if fig: savefig(fig, "07_paired_ff")

In [ ]:
# Cartes de résidus (visualization.plot_residuals) : ff=0 vs ff=1, autres features inchangées
COMPARE = ("model_ff0_as0_ts0_alw0", "model_ff1_as0_ts0_alw0")
T_PLOT = 0.5   # [s]

for n in COMPARE:
    if n not in RES:
        print("checkpoint absent pour", n)
        continue
    m = load_run_model(n)
    fig = plot_residuals(m, t=T_PLOT, fields=("continuity", "momentum_x", "momentum_y"), log=True)
    fig.suptitle(f"{short(n)} — résidus à t = {T_PLOT} s", y=1.02)
    savefig(fig, f"08_residual_maps_{short(n)}")
    del m

## 7. Segmentation temporelle (`ts`)

### 7.1 Comportement fenêtre par fenêtre
Pour les runs `ts=1` : loss `core` en fin de fenêtre, et **saut** à l'ouverture de la fenêtre suivante
(`début(w+1) / fin(w)`) — un saut élevé signale un transfert difficile entre fenêtres (oubli, raccord imparfait malgré le replay).

In [ ]:
def window_table(df, n=10):
    rows = []
    for w, g in df.groupby("window"):
        rows.append({"window": int(w), "t_left": g["t_left"].iloc[0], "t_right": g["t_right"].iloc[0],
                     "epochs": len(g), "core_start": g["core"].iloc[:n].mean(), "core_end": g["core"].iloc[-n:].mean()})
    t = pd.DataFrame(rows).set_index("window")
    t["jump_to_next"] = t["core_start"].shift(-1) / t["core_end"]
    return t


ts_runs = [n for n in HIST if RUNS.loc[n, "ts"] == 1]
if ts_runs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for n in ts_runs:
        wt = window_table(HIST[n])
        axes[0].semilogy(wt.index, wt["core_end"], marker="o", label=short(n))
        axes[1].semilogy(wt.index[:-1], wt["jump_to_next"].iloc[:-1], marker="s", label=short(n))
    axes[0].set(title="Loss core en fin de fenêtre", xlabel="Fenêtre")
    axes[1].set(title="Saut de loss au changement de fenêtre (×)", xlabel="Fenêtre w → w+1")
    axes[1].axhline(1, color="k", lw=0.8)
    axes[0].legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, "09_temporal_windows")
    display(window_table(HIST[ts_runs[0]]))

### 7.2 Dérive temporelle hold-out : `ts=0` vs `ts=1`
Résidu physique en fonction de `t` (moyenne géométrique sur les 8 runs de chaque groupe, bande = min/max).

In [ ]:
if RES:
    fig, ax = plt.subplots(figsize=(12, 5))
    for val, col in ((0, "C0"), (1, "C3")):
        names = [n for n in RES if RUNS.loc[n, "ts"] == val]
        if not names:
            continue
        stack = np.log10(np.stack([sum(RES[n][k] for k in PHYS) for n in names]))
        t = RES[names[0]]["t"]
        ax.plot(t, 10 ** stack.mean(0), color=col, label=f"ts={val} (n={len(names)})")
        ax.fill_between(t, 10 ** stack.min(0), 10 ** stack.max(0), color=col, alpha=0.2)
    if ts_runs:
        for tl in HIST[ts_runs[0]].groupby("window")["t_left"].first().iloc[1:]:
            ax.axvline(tl, color="k", lw=0.5, ls="--", alpha=0.5)
    ax.set_yscale("log")
    ax.set_xlabel("t [s]")
    ax.set_ylabel("Résidu physique moyen²")
    ax.legend()
    ax.set_title("Dérive temporelle du résidu (traits pointillés : frontières de fenêtres)")
    fig.tight_layout()
    savefig(fig, "10_ts_drift")

fig = plot_paired("ts", ["res_phys", "osc"])
if fig: savefig(fig, "11_paired_ts")

## 8. Échantillonnage adaptatif (`as`) — points enregistrés

Source : `adaptive_history.npz` (points **adimensionnés** `(M, N, 3)` = (x*, y*, t*), scores, résidus locaux, `alpha`) et `adaptive_history.json` (résumé).
`M` = nombre de rafraîchissements (tous les `sampling_update_interval` epochs).

In [ ]:
def load_adaptive(name):
    d = RUNS.loc[name, "dir"]
    npz, js = d / "adaptive_history.npz", d / "adaptive_history.json"
    if not npz.exists():
        return None
    z = np.load(npz)
    data = {k: z[k] for k in z.files}
    if js.exists():
        data["summary"] = pd.DataFrame(json.loads(js.read_text(encoding="utf-8")))
    return data


as_runs = [n for n in HIST if RUNS.loc[n, "as"] == 1]
ADA = {n: a for n in as_runs if (a := load_adaptive(n)) is not None}
print(f"{len(ADA)} runs avec historique de points")

if ADA:
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.2))
    for n, a in ADA.items():
        s = a["summary"]
        for ax, col in zip(axes, ["mean_score", "mean_continuity", "mean_vof", "mean_momentum"]):
            ax.semilogy(s["epoch"], s[col], label=short(n), lw=1.2)
    for ax, t in zip(axes, ["Score moyen des points retenus", "Continuité", "VOF", "Quantité de mouvement"]):
        ax.set_title(t)
        ax.set_xlabel("Epoch")
    axes[0].legend(fontsize=6)
    fig.tight_layout()
    savefig(fig, "12_adaptive_scores")

In [ ]:
# Fraction de points retenus proches de l'interface eau/air (0.05 < alpha < 0.95)
if ADA:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for n, a in ADA.items():
        frac = ((a["alpha"] > 0.05) & (a["alpha"] < 0.95)).mean(axis=1)
        ax.plot(a["epochs"], frac, label=short(n), lw=1.2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Fraction de points à l'interface")
    ax.set_title("Concentration de l'échantillonnage adaptatif sur l'interface")
    ax.legend(fontsize=6, ncol=2)
    fig.tight_layout()
    savefig(fig, "13_adaptive_interface_fraction")

In [ ]:
# Détail d'un run : nuages de points (début / milieu / fin) + densité temporelle
SAMPLING_RUN = "model_ff1_as1_ts0_alw0"

if SAMPLING_RUN in ADA:
    a = ADA[SAMPLING_RUN]
    M = a["points"].shape[0]
    picks = sorted({0, M // 2, M - 1})

    fig, axes = plt.subplots(1, len(picks), figsize=(6 * len(picks), 4.5), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, picks):
        P, S = a["points"][i], np.log10(np.maximum(a["scores"][i], 1e-16))
        sc = ax.scatter(P[:, 0], P[:, 1], c=S, s=4, cmap="magma")
        ax.set_title(f"epoch {int(a['epochs'][i])}  (fenêtre {int(a['windows'][i])})")
        ax.set_xlabel("x*")
        ax.set_aspect("equal", adjustable="box")
        fig.colorbar(sc, ax=ax, label="log10 score", fraction=0.046)
    axes[0].set_ylabel("y*")
    fig.suptitle(f"Points adaptatifs — {short(SAMPLING_RUN)}", y=1.02)
    fig.tight_layout()
    savefig(fig, "14_adaptive_points")

    # densité en t* au fil des rafraîchissements
    tmin, tmax = a["points"][..., 2].min(), a["points"][..., 2].max()
    bins = np.linspace(tmin, tmax, 41)
    H = np.stack([np.histogram(a["points"][i, :, 2], bins=bins)[0] for i in range(M)], axis=1)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    im = ax.imshow(H, aspect="auto", origin="lower", cmap="viridis",
                   extent=[a["epochs"][0], a["epochs"][-1], tmin, tmax])
    ax.set_xlabel("Epoch")
    ax.set_ylabel("t*")
    ax.set_title("Où l'échantillonnage adaptatif place-t-il ses points en temps ?")
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="nb de points")
    fig.tight_layout()
    savefig(fig, "15_adaptive_time_density")
elif ADA:
    print("Choisir SAMPLING_RUN parmi :", list(ADA))

In [ ]:
fig = plot_paired("as", ["core_final", "res_phys", "osc"])
if fig: savefig(fig, "16_paired_as")

## 9. Poids adaptatifs GradNorm (`alw`) — poids enregistrés

Source : `adaptive_weights.json` (poids de chaque terme à chaque epoch) et `adaptive_weights_final.json`.

### 9.1 Les poids enregistrés sont-ils réellement appliqués en continu ?

Dans `train.py`, le dictionnaire `weights` est **réinitialisé aux poids fixes de `Config` à chaque epoch**, puis mis à jour par
`weight_balancer.update(...)` **uniquement** si `local_epoch % weight_update_interval == 0` (et hors epochs de rafraîchissement des points).
Conséquence possible : les poids GradNorm ne s'appliqueraient qu'à ~1 epoch sur `weight_update_interval`, les autres epochs revenant aux poids fixes
(et le L-BFGS final partirait de `frozen_weights = dict(weights)` de la dernière epoch). Le diagnostic ci-dessous le vérifie sur vos données :
si la fraction d'epochs dont les poids diffèrent des poids fixes ≈ `1/weight_update_interval`, c'est bien le cas.

In [ ]:
FIXED_MAP = {"continuity": "w_continuity", "vof": "w_vof", "momentum_x": "w_momentum",
             "momentum_y": "w_momentum", "wall_u": "w_wall", "wall_v": "w_wall"}
WEIGHT_NAMES = list(FIXED_MAP)


def load_weights(name):
    p = RUNS.loc[name, "dir"] / "adaptive_weights.json"
    return pd.DataFrame(json.loads(p.read_text(encoding="utf-8"))) if p.exists() else None


alw_runs = [n for n in HIST if RUNS.loc[n, "alw"] == 1]
WTS = {n: w for n in alw_runs if (w := load_weights(n)) is not None}
print(f"{len(WTS)} runs avec historique de poids")

rows = []
for n, w in WTS.items():
    cfg = CFGS.get(n, {})
    fixed = {k: cfg.get(v) for k, v in FIXED_MAP.items()}
    if any(v is None for v in fixed.values()):
        continue
    differs = np.column_stack([(w[k] - fixed[k]).abs() > 1e-12 for k in WEIGHT_NAMES]).any(axis=1)
    interval = cfg.get("weight_update_interval")
    rows.append({"run": short(n), "weight_update_interval": interval,
                 "attendu si persistants": 1.0,
                 "attendu si appliqués 1 epoch/intervalle": (1 / interval) if interval else np.nan,
                 "fraction mesurée (poids ≠ fixes)": differs.mean()})
pd.DataFrame(rows).set_index("run") if rows else print("config.json sans les poids fixes (w_*) : diagnostic impossible")

### 9.2 Évolution des poids

In [ ]:
if WTS:
    n_runs = len(WTS)
    ncols = min(4, n_runs)
    nrows = int(np.ceil(n_runs / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4 * nrows), sharey=True, squeeze=False)
    for ax, (n, w) in zip(axes.ravel(), WTS.items()):
        for k in WEIGHT_NAMES:
            ax.semilogy(w["epoch"], w[k], lw=1.0, label=k)
        if RUNS.loc[n, "ts"] == 1:
            for e in w.groupby("window")["epoch"].min().iloc[1:]:
                ax.axvline(e, color="k", lw=0.4, alpha=0.4)
        ax.set_title(short(n), fontsize=9)
        ax.set_xlabel("Epoch")
    axes.ravel()[0].legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, "17_weights_evolution")

In [ ]:
# Poids finaux (adaptive_weights_final.json) — échelle log10
FIN = {}
for n in alw_runs:
    p = RUNS.loc[n, "dir"] / "adaptive_weights_final.json"
    if p.exists():
        FIN[short(n)] = json.loads(p.read_text(encoding="utf-8"))
if FIN:
    F = pd.DataFrame(FIN).T[WEIGHT_NAMES]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(F) + 2))
    im = ax.imshow(np.log10(F.values.astype(float)), cmap="coolwarm", aspect="auto")
    ax.set_xticks(range(len(WEIGHT_NAMES)), WEIGHT_NAMES, rotation=30)
    ax.set_yticks(range(len(F)), F.index)
    for i in range(F.shape[0]):
        for j in range(F.shape[1]):
            ax.text(j, i, f"{F.values[i, j]:.2g}", ha="center", va="center", fontsize=8)
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="log10(poids)")
    ax.set_title("Poids GradNorm finaux")
    fig.tight_layout()
    savefig(fig, "18_weights_final")

fig = plot_paired("alw", ["core_final", "res_phys", "osc"])
if fig: savefig(fig, "19_paired_alw")

## 10. Comparaison à une référence OpenFOAM (optionnel mais recommandé)

Renseigner `OF_DATA` (`.npz` avec `X`, `time`, `alpha`, éventuellement `u`, `v`, `p`). Erreur L2 relative par champ, et en fonction du temps.

In [ ]:
OF_DATA = None   # ex. "data/openfoam.npz"
OF_FIELDS = ("alpha", "u", "v", "p")

OF_ERR, OF_ERR_T = {}, {}
if OF_DATA is not None:
    with np.load(OF_DATA) as z:
        of = {k: z[k] for k in z.files}
    for name in RUNS.index:
        if name not in CFGS or not (RUNS.loc[name, "dir"] / "checkpoint_final.pt").exists():
            continue
        model = load_run_model(name)
        sinn = build_sinn_data_from_model(model, of)
        errs, errs_t = {}, {}
        for f in OF_FIELDS:
            if f in of and f in sinn:
                pred = np.asarray(sinn[f])
                ref = np.asarray(of[f]).reshape(pred.shape)
                errs[f"err_{f}"] = np.linalg.norm(pred - ref) / max(np.linalg.norm(ref), 1e-12)
                errs_t[f] = np.linalg.norm(pred - ref, axis=1) / np.maximum(np.linalg.norm(ref, axis=1), 1e-12)
        OF_ERR[short(name)] = errs
        OF_ERR_T[name] = (np.asarray(of["time"]).ravel(), errs_t)
        del model
    OF_DF = pd.DataFrame(OF_ERR).T
    METRICS = METRICS.drop(columns=[c for c in OF_DF.columns if c in METRICS.columns]).join(OF_DF)
    display(METRICS[FACTORS + list(OF_DF.columns)].sort_values(OF_DF.columns[0]))

    fig, ax = plt.subplots(figsize=(12, 5))
    for n, (t, e) in OF_ERR_T.items():
        if "alpha" in e:
            ax.semilogy(t, e["alpha"], label=short(n), lw=1.2)
    ax.set_xlabel("t [s]")
    ax.set_ylabel("Erreur L2 relative sur alpha")
    ax.legend(ncol=2, fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
    fig.tight_layout()
    savefig(fig, "20_openfoam_alpha_error_vs_time")
else:
    print("OF_DATA non renseigné : section ignorée.")

## 11. Synthèse : classement des 16 configurations et export

In [ ]:
# Si OpenFOAM est disponible, il fait foi ; sinon résidu hold-out ; sinon loss core
RANK_BY = next((c for c in ["err_alpha", "res_phys", "core_final"] if c in METRICS.columns), "core_final")
cols = FACTORS + [c for c in ["err_alpha", "res_phys", "core_final", "decades_gained", "osc", "ep_1dec"] if c in METRICS.columns]
SUMMARY = METRICS[cols].sort_values(RANK_BY)
SUMMARY["rang"] = np.arange(1, len(SUMMARY) + 1)
print(f"Classement selon : {RANK_BY}")
display(SUMMARY)

SUMMARY.to_csv(FIG_DIR / "summary_16_configs.csv")
EFF.to_csv(FIG_DIR / "factorial_effects.csv")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([short(i) if i.startswith("model_") else i for i in SUMMARY.index[::-1]],
        SUMMARY[RANK_BY].values[::-1], color="C0")
ax.set_xscale("log")
ax.set_xlabel(RANK_BY)
ax.set_title("Classement des 16 configurations")
fig.tight_layout()
savefig(fig, "21_ranking")

### Pistes pour consolider les conclusions
* **Répéter avec ≥ 3 graines** par configuration (`cfg.seed`) pour obtenir des barres d'erreur sur les effets (section 5).
* Comparer à **budget de calcul égal** (temps mur ou nombre d'évaluations de résidu) : l'échantillonnage adaptatif (pool de `n_candidate_points`) et GradNorm (gradients supplémentaires) coûtent plus cher par epoch ; `history` ne contient pas le temps mur — à ajouter dans `train.py`.
* Vérifier le point de la section 9.1 (poids GradNorm non persistants) avant d'interpréter l'effet de `alw`.